In [1]:
from google.colab import files
uploaded = files.upload()

Saving expenses_large.csv.txt to expenses_large.csv.txt


In [2]:


from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("ExpenseAnalysis").getOrCreate()

In [4]:
import os
os.listdir()

['.config', 'expenses_large.csv.txt', 'sample_data']

In [5]:
import os

os.rename("expenses_large.csv.txt", "expenses_large.csv")

In [6]:
!ls /content

expenses_large.csv  sample_data


In [7]:
df = spark.read.csv("/content/expenses_large.csv", header=True, inferSchema=True)
df.show()

+-------+----------+-------------+------+
|user_id|      date|     category|amount|
+-------+----------+-------------+------+
|      1|2026-01-01|         Food|   500|
|      1|2026-01-02|    Transport|   200|
|      1|2026-01-03|     Shopping|  1500|
|      1|2026-01-04|        Bills|   800|
|      1|2026-01-05|         Food|   600|
|      2|2026-01-01|         Food|   400|
|      2|2026-01-02|     Shopping|  2000|
|      2|2026-01-03|    Transport|   300|
|      2|2026-01-04|        Bills|  1000|
|      2|2026-01-05|         Food|   350|
|      3|2026-01-01|Entertainment|   700|
|      3|2026-01-02|         Food|   500|
|      3|2026-01-03|     Shopping|  2500|
|      3|2026-01-04|    Transport|   200|
|      3|2026-01-05|        Bills|  1200|
|      4|2026-01-01|         Food|   450|
|      4|2026-01-02|     Shopping|  1800|
|      4|2026-01-03|    Transport|   300|
|      4|2026-01-04|        Bills|   900|
|      4|2026-01-05|Entertainment|  1000|
+-------+----------+-------------+

In [8]:
from pyspark.sql.functions import month, sum

df = df.withColumn("month", month("date"))

monthly_spend = df.groupBy("user_id", "month") \
    .agg(sum("amount").alias("total_spend"))

monthly_spend.show()

+-------+-----+-----------+
|user_id|month|total_spend|
+-------+-----+-----------+
|      3|    1|       5100|
|      2|    2|       4290|
|      1|    2|       3840|
|      1|    1|       3600|
|      2|    1|       4050|
|      3|    2|       5380|
|      5|    2|      19550|
|      5|    1|      18500|
|      4|    2|       4750|
|      4|    1|       4450|
+-------+-----+-----------+



In [9]:
unusual = df.filter(df["amount"] > 5000)
unusual.show()

+-------+----------+--------+------+-----+
|user_id|      date|category|amount|month|
+-------+----------+--------+------+-----+
|      5|2026-01-01|    Food|  6000|    1|
|      5|2026-01-02|Shopping|  7000|    1|
|      5|2026-02-01|    Food|  6500|    2|
|      5|2026-02-02|Shopping|  7200|    2|
+-------+----------+--------+------+-----+



In [10]:
top_users = df.groupBy("user_id") \
    .agg(sum("amount").alias("total_spent")) \
    .orderBy("total_spent", ascending=False)

top_users.show()

+-------+-----------+
|user_id|total_spent|
+-------+-----------+
|      5|      38050|
|      3|      10480|
|      4|       9200|
|      2|       8340|
|      1|       7440|
+-------+-----------+

